# 01 — Baseline exploration

First look at the raw benchmark data after Phase 1 collection.
Verify thermal behaviour, latency distributions, and CV floor.

In [ ]:
import json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

In [ ]:
# Load all raw JSON files
runs = []
for path in sorted(glob.glob('data/raw/*.json')):
    with open(path) as f:
        runs.append(json.load(f))
print(f'Loaded {len(runs)} runs')

In [ ]:
# Summary table
rows = []
for r in runs:
    rows.append({
        'model': r.get('model_name', '?'),
        'compute_unit': r.get('compute_unit', '?'),
        'median_us': r.get('median_us'),
        'p99_us': r.get('p99_us'),
        'cv_pct': r.get('cv_pct'),
        'size_mb': r.get('model_size_mb'),
        'aborted_thermal': r.get('run_aborted_thermal', False),
    })
df = pd.DataFrame(rows)
print(df.to_string())

In [ ]:
# Latency distribution for each run
fig, axes = plt.subplots(1, min(len(runs), 4), figsize=(14, 4))
if len(runs) == 1: axes = [axes]
for ax, run in zip(axes, runs[:4]):
    lats = run.get('latencies_us', [])
    ax.hist(lats, bins=40, color='#185FA5', alpha=0.75, edgecolor='white', linewidth=0.3)
    ax.axvline(run.get('median_us', 0), color='#D85A30', linestyle='--', linewidth=1.5)
    ax.set_title(f"{run.get('model_name','?')[:30]}\n{run.get('compute_unit','?')}", fontsize=8)
    ax.set_xlabel('Latency (µs)', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Thermal samples for first run with thermal data
for run in runs:
    samples = run.get('thermal_samples', [])
    if samples:
        t0 = samples[0]['timestamp_s']
        ts = [s['timestamp_s'] - t0 for s in samples]
        temps = [s['celsius'] for s in samples]
        plt.figure(figsize=(9, 3))
        plt.plot(ts, temps, color='#D85A30', linewidth=1.5)
        plt.axhline(45, color='gray', linestyle='--', linewidth=1)
        plt.xlabel('Time (s)')
        plt.ylabel('Skin temp (°C)')
        plt.title(f"Thermal — {run.get('model_name', '?')}")
        plt.tight_layout()
        plt.show()
        break

In [ ]:
# CV floor check — if CV > 10%, the measurement setup has noise issues
high_cv = df[df['cv_pct'] > 10.0]
if high_cv.empty:
    print('All runs CV <= 10% — measurement setup is stable')
else:
    print(f'{len(high_cv)} runs with CV > 10% (may need investigation):')
    print(high_cv[['model', 'compute_unit', 'cv_pct']].to_string())